In [ ]:
import pandas as pd
import looker_sdk
from io import StringIO

In [ ]:
# Load the Looker Python SDK tools.
import looker_sdk

# Path to the `looker.ini` file we setup.


CREDENTIALS_PATH = "utils\\looker.ini"  # <--- Change


# Initializing the Looker SDK.


sdk = looker_sdk.init40(CREDENTIALS_PATH)

# Retrieve my user information from Looker.
my_user = sdk.me()

# Display my username.
my_user["display_name"]

In [ ]:
#############
# Parameters
#############
RESULT_FORMAT = "csv"

#############
# Send Query
#############

# Define your query
body = {
    "model": "oahu_connection_model",
    "view": "base",
    "fields": [
        "clients.full_name",
        "clients.unique_identifier",
        "clients.id",
        "enrollments.ref_household",
        "household_makeup.household_type",
        "entry_screen.relationship_to_hoh",
        "clients.birth_date",
        "entry_screen.age",
        "clients.age",
        "agencies.name",
        "programs.name",
        "programs.project_type_code",
        "enrollments.id",
        "enrollments.start_date",
        "enrollments.end_date",
        "enrollments.days_since_start",
        "household_move_in_date.move_in_date",
        "members.assigned_staff",
        "static_demographics.veteran_text",
        "services.service_added_date",
        "client_notes_enrollment_level.note",
        "client_notes.id",
        "client_notes.added_date",
        "screens.name",
        "entry_screen.head_of_household",
        "enrollments.still_in_program",
        "client_assessments.assessment_date",
        "enrollments.added_date",
        "statusannual_assessment_count",
        "services.count",
        "client_notes_enrollment_level.count",
    ],
    "filters": {
        "enrollments.date_filter": "1 quarters",
        "programs.status": "Active",
    },
    "limit": 100000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
df_1 = pd.read_csv(StringIO(result))

In [ ]:
body = {
    "model": "oahu_connection_model",
    "view": "client_model",
    "fields": [
        "clients.unique_identifier",
        "referrals.reassigned_date_date",
        "referrals.Currently_On_Queue",
        "referrals_referral_community_queues.name",
        "client_model.deleted",
        "enrollments.deleted",
    ],
    "filters": {
        "enrollments.date_filter": "1 quarters",
        "programs.status": "Active",
        "client_model.deleted": "No",
        "enrollments.deleted": "No",
    },
    "limit": 100000,
}

result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
df_2 = pd.read_csv(StringIO(result))

In [ ]:
rename_map = {"Unique Identifier": "Clients Unique Identifier"}

In [ ]:
df_2 = df_2.rename(columns=rename_map)

In [ ]:
df_3 = pd.merge(
    df_1,
    df_2,
    "left",
    on="Clients Unique Identifier",
)

In [ ]:
df_2[df_2["Clients Unique Identifier"].duplicated()]

In [ ]:
df_1[df_1["Clients Unique Identifier"].duplicated()]

In [ ]:
df_3[df_3["Clients Unique Identifier"].duplicated()]

In [ ]:
from ds_sql_connections import GsAutomationReliefAssistanceConnection
import os
import json

main_directory_path = str(os.getcwd())
# Path to the config file
config_path = main_directory_path + "\\utils\\config_partners_in_care.json"
# creating the config object to pull the authentication credentials later in various calls.
with open(config_path, "r", encoding="utf-8") as file:
    config_object = json.load(file)

# sql connection credentials
sql_username = config_object["Josh"]["username"]
sql_password = config_object["Josh"]["password"]
conn = GsAutomationReliefAssistanceConnection(sql_username, sql_password, auth=True)
conn.insert_with_progress(
    df=df_3,
    schema="dbo",
    tbl_nm="partners_in_care_active_projects",
    dtypes={},
)

In [ ]:
df_1.to_excel("partners_in_care_active_projects.xlsx")

In [ ]:
look = sdk.look("92496")

In [ ]:
filter = look.query["filters"]

In [ ]:
columns = look.query["fields"]

In [ ]:
# Define your query
body = {
    "model": "oahu_connection_model",
    "view": "base",
    "fields": columns,
    "filters": filter,
    "limit": 10000,
}

################
# Get Response
################

# Run the inline query
result = sdk.run_inline_query(result_format=RESULT_FORMAT, body=body)

# Convert to dataframe.
df = pd.read_csv(StringIO(result))

In [ ]:
df

In [ ]:
import json
import sys
from typing import cast, Dict, List, Union

import looker_sdk
from looker_sdk import models40 as models, error

In [ ]:
def main() -> None:
    look_id = "92488"
    filter_args = iter(sys.argv[2:])
    filters: Dict[str, str] = {}

    if not (look_id and len(sys.argv[2:]) > 0 and len(sys.argv[2:]) % 2 == 0):
        raise Exception(
            "Please provide: <lookId> <filter_1> <filter_value_1> "
            "<filter_2> <filter_value_2> ..."
        )

    for filter_name in filter_args:
        filters[filter_name] = next(filter_args)

    query = get_look_query(int(look_id))
    results = run_query_with_filter(query, filters)

    print(f"Query results with filters={filters}:\n{results}", end="\n\n")


def get_look_query(id: int) -> models.Query:
    """Returns the query associated with a given look id."""
    try:
        look = sdk.look(id)
    except error.SDKError:
        raise Exception(f"Error getting Look {id}")
    else:
        query = look.query
    return query


TJson = List[Dict[str, Union[str, int, float, bool, None]]]


def run_query_with_filter(query: models.Query, filters: Dict[str, str]) -> TJson:
    """Runs the specified query with the specified filters."""
    request = create_query_request(query, filters)
    try:
        json_ = sdk.run_inline_query("json", request, cache=False)
    except error.SDKError:
        raise Exception("Error running query")
    else:
        json_resp = cast(TJson, json.loads(json_))
    return json_resp


def create_query_request(q: models.Query, filters: Dict[str, str]) -> models.WriteQuery:
    return models.WriteQuery(
        model=q.model,
        view=q.view,
        fields=q.fields,
        pivots=q.pivots,
        fill_fields=q.fill_fields,
        filters=filters,
        sorts=q.sorts,
        limit=q.limit,
        column_limit=q.column_limit,
        total=q.total,
        row_total=q.row_total,
        subtotals=q.subtotals,
        dynamic_fields=q.dynamic_fields,
        query_timezone=q.query_timezone,
    )

In [ ]:
main()